# Ernesto Investing AI — Notebook 06: API FastAPI + ngrok

Levanta el backend completo (`backend/main.py`, con los 18 endpoints:
autenticacion, mercado, predicciones, metricas, ranking, actividad,
historial y sentimiento) directamente desde Google Colab, y lo expone
a Internet con ngrok para que el frontend pueda consumirlo.

**Requisitos previos:**
1. Haber ejecutado al menos los Notebooks 01 y 02 (para que haya datos).
2. Tener el archivo `MONGO_URI` configurado como Secret de Colab.
3. Tener una cuenta gratuita en https://ngrok.com y su authtoken,
   configurado como Secret de Colab bajo el nombre `NGROK_AUTH_TOKEN`.
4. Subir la carpeta `backend/` completa a este entorno de Colab
   (arrastrar la carpeta al panel de archivos, o clonar el repositorio
   de GitHub con `!git clone`).

## 1. Instalacion de librerias

In [ ]:
!pip install fastapi uvicorn "pymongo[srv]" pydantic[email] PyJWT bcrypt pyngrok nest-asyncio --quiet

## 2. Subir o clonar el codigo del backend

Opcion A (recomendada): clonar el repositorio de GitHub donde ya
subiste el proyecto completo.

Opcion B: subir manualmente la carpeta `backend/` usando el panel
de archivos de Colab (icono de carpeta en la barra lateral izquierda).

In [ ]:
# Opcion A: clonar desde GitHub (reemplaza con la URL de tu repositorio)
# !git clone https://github.com/tu-usuario/ernesto-investing-ai.git
# %cd ernesto-investing-ai

import os
assert os.path.isdir("backend"), (
    "No se encontro la carpeta backend/. Clona el repositorio o sube "
    "la carpeta manualmente antes de continuar."
)
print("Carpeta backend/ encontrada correctamente.")

## 3. Configuracion de variables de entorno

El backend lee `MONGO_URI` y `JWT_SECRET` desde variables de entorno
(ver `backend/config.py`). En Colab, se toman de los Secrets y se
inyectan al entorno del proceso antes de importar la app.

In [ ]:
import os
from google.colab import userdata

os.environ["MONGO_URI"] = userdata.get("MONGO_URI")
os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")  # si no existe el Secret, define uno propio
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

print("Variables de entorno configuradas.")

## 4. Configuracion de ngrok

In [ ]:
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()  # cierra cualquier tunel previo abierto en esta sesion

print("ngrok configurado correctamente.")

## 5. Levantar la API con uvicorn en un hilo en segundo plano

`nest_asyncio` permite ejecutar el loop de asyncio de uvicorn dentro
del loop que ya usa Jupyter/Colab (sin esto, uvicorn no puede
arrancar dentro de un notebook).

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
import sys

sys.path.insert(0, os.getcwd())
nest_asyncio.apply()

from backend.main import app

def _correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

hilo_servidor = threading.Thread(target=_correr_servidor, daemon=True)
hilo_servidor.start()

time.sleep(3)  # da tiempo a que el servidor termine de arrancar
print("Servidor FastAPI iniciado en el puerto 8000.")

## 6. Exponer el servidor a Internet con ngrok

In [ ]:
tunel = ngrok.connect(8000)
URL_PUBLICA = tunel.public_url

print("=" * 60)
print("API disponible publicamente en:")
print(URL_PUBLICA)
print("Documentacion interactiva (Swagger UI):", URL_PUBLICA + "/docs")
print("=" * 60)
print()
print("Copia esta URL y pegala en el portal (index.html) del frontend,")
print("o configurala en la app de Streamlit.")

## 7. Verificacion rapida

Prueba el endpoint de salud y un endpoint real de datos para
confirmar que la API responde correctamente antes de conectarla
al frontend.

In [ ]:
import requests

respuesta_salud = requests.get(f"{URL_PUBLICA}/api/salud").json()
print("Salud del sistema:", respuesta_salud)

print()
respuesta_mercado = requests.get(f"{URL_PUBLICA}/api/mercado/BVN?periodo=5").json()
print("Ejemplo /api/mercado/BVN:", respuesta_mercado)

## 8. Mantener el servidor activo

Esta celda mantiene el notebook "vivo" para que el tunel de ngrok
no se cierre. **No la ejecutes si solo quieres probar la API
brevemente** (las celdas anteriores ya la dejan corriendo en segundo
plano); ejecutala cuando vayas a hacer la demo en vivo o dejar el
sistema activo mientras trabajas en el frontend.

In [ ]:
import time

print("Servidor activo. Presiona el boton de Detener (cuadrado) para cerrar el tunel.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    ngrok.kill()
    print("Tunel cerrado.")